# Module 04: Data Modeling

**Program:** Quintrix Jr. Data Engineer Training
**Duration:** 2 hours
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Why Data Modeling? (OLTP → OLAP) | 10 min |
| 2 | Star Schema — The Gold Standard | 15 min |
| 3 | Building Dimension Tables | 20 min |
| 4 | Slowly Changing Dimensions (SCD) | 20 min |
| 5 | Building the Fact Table | 15 min |
| 6 | Star Schema Queries | 10 min |
| 7 | Partitioning & Bucketing | 10 min |
| 8 | Challenge Lab | 10 min |
| 9 | Key Takeaways | 5 min |
| 10 | Homework & Preview | 5 min |

---

## 1. Why Data Modeling? (OLTP → OLAP)

**One-line answer:** Data modeling transforms messy operational data into clean, queryable structures optimized for analysis.


> **Analogy:** OLTP is the cash register — it records one sale at a time, fast and accurate. OLAP is the end-of-month report — it reads every transaction to find patterns and totals. Our call center database (M02–M03) is OLTP. Today we transform it into OLAP.

| | OLTP | OLAP |
|---|------|------|
| **Purpose** | Run the business (transactions) | Analyze the business (reporting) |
| **Schema** | Normalized (3NF — Third Normal Form, a database design with minimal data duplication, covered in M00 Section 16a) — minimal redundancy | Denormalized (star/snowflake — snowflake is a variation of star schema where dimensions have sub-dimensions; we focus on star schema in this module) — optimized for reads |
| **Queries** | Short, targeted (INSERT/UPDATE one row) | Wide, aggregating (SUM/AVG across millions) |
| **Users** | Applications, APIs | Analysts, dashboards, ML models |
| **Example** | `INSERT INTO orders ...` | `SELECT campaign, SUM(revenue) GROUP BY ...` |

Our call center database (M02–M03) is **OLTP** — it stores individual calls, orders, and payments as they happen. Today we transform it into **OLAP** — a structure designed for fast, intuitive analytics.

> **Medallion architecture connection:** Bronze (raw ingestion) → Silver (cleaned/validated) → **Gold (modeled for analytics) = star schema**. Today we build the Gold layer.

---

## 0. Hello World: What Is Data Modeling?

Before we write a single schema, let's see the problem data modeling solves — going from a messy all-in-one table to a clean, normalized design. This is the foundation everything else in this module builds on.

In [ ]:
# =============================================================================
# HELLO WORLD: What IS Data Modeling?
# =============================================================================
# Before we write a single schema, let's see THE problem data modeling solves.
# We'll go from "everything in one messy table" to a clean, organized design.
# =============================================================================

import sqlite3

# Step 1: The naive approach — store everything in ONE table (like a spreadsheet)
# WHY THIS IS BAD: Every time a customer places a second order, you copy their
#                  name, city, and email again. That's wasted space + data drift.
#                  If the customer changes their city, you must update EVERY row.

demo = sqlite3.connect(":memory:")
demo.execute('''
    CREATE TABLE all_in_one (
        order_id    TEXT,
        customer_name TEXT,   -- repeated for EVERY order by this customer
        customer_city TEXT,   -- repeated for EVERY order by this customer
        product_name  TEXT,   -- repeated for EVERY order of this product
        product_price REAL,   -- repeated for EVERY order of this product
        order_total   REAL
    )
''')

# BEGINNER NOTE: Look at how "Alice" and "Widget A" appear multiple times.
#                In a real system, one customer might have 10,000 orders.
#                That's 10,000 copies of their name — pure waste.
demo.executemany("INSERT INTO all_in_one VALUES (?,?,?,?,?,?)", [
    ("ORD-001", "Alice",   "Detroit",  "Widget A", 29.99, 29.99),
    ("ORD-002", "Alice",   "Detroit",  "Widget B", 49.99, 49.99),  # Alice repeated!
    ("ORD-003", "Bob",     "Chicago",  "Widget A", 29.99, 29.99),
    ("ORD-004", "Alice",   "Detroit",  "Widget A", 29.99, 59.98),  # Alice again!
    ("ORD-005", "Bob",     "Chicago",  "Widget B", 49.99, 49.99),
])

print("=== The Messy Spreadsheet Approach (Denormalized) ===")
print(f"{'order_id':<8}  {'customer_name':<12}  {'city':<9}  {'product_name':<10}  {'price':>7}  {'total':>8}")
print("-" * 68)
for r in demo.execute("SELECT * FROM all_in_one"):
    print(f"{r[0]:<8}  {r[1]:<12}  {r[2]:<9}  {r[3]:<10}  ${r[4]:>6.2f}  ${r[5]:>6.2f}")

# Highlight the problem: "Detroit" is stored 3 times for Alice
alice_rows = demo.execute(
    "SELECT COUNT(*) FROM all_in_one WHERE customer_name='Alice'"
).fetchone()[0]
print(f"\nAlice's city 'Detroit' is stored {alice_rows} times.")
print("If Alice moves, you must UPDATE every single one of those rows.")
print("Miss one? Now your data is inconsistent.")

# ── Step 2: Normalize — split into proper tables ──────────────────────────────
# WHY: Each fact lives in exactly ONE place. Change Alice's city once — done.
#      This is the foundation of OLTP (Online Transaction Processing) design.
print()
print("=== The Normalized Approach (OLTP) ===")

demo.execute("CREATE TABLE customers (cust_id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
demo.execute("CREATE TABLE products  (prod_id INTEGER PRIMARY KEY, name TEXT, price REAL)")
demo.execute("CREATE TABLE orders    (order_id TEXT PRIMARY KEY, cust_id INTEGER, prod_id INTEGER, total REAL)")

# Each customer stored ONCE — a single source of truth
demo.executemany("INSERT INTO customers VALUES (?,?,?)", [
    (1, "Alice", "Detroit"),   # Alice stored exactly once
    (2, "Bob",   "Chicago"),   # Bob stored exactly once
])
# Each product stored ONCE
demo.executemany("INSERT INTO products VALUES (?,?,?)", [
    (1, "Widget A", 29.99),    # Widget A stored exactly once
    (2, "Widget B", 49.99),
])
# Orders reference customers and products by ID (Foreign Key = FK)
# FK = a column that points to the Primary Key (PK) of another table
demo.executemany("INSERT INTO orders VALUES (?,?,?,?)", [
    ("ORD-001", 1, 1, 29.99),  # cust_id=1 means Alice, prod_id=1 means Widget A
    ("ORD-002", 1, 2, 49.99),
    ("ORD-003", 2, 1, 29.99),
    ("ORD-004", 1, 1, 59.98),
    ("ORD-005", 2, 2, 49.99),
])

print("customers: each person stored once")
for r in demo.execute("SELECT * FROM customers"):
    print(f"  {r}")
print("\nproducts: each product stored once")
for r in demo.execute("SELECT * FROM products"):
    print(f"  {r}")
print("\norders: just IDs + total (no repeated names)")
for r in demo.execute("SELECT * FROM orders"):
    print(f"  {r}")

# ── Step 3: The Insight — OLTP ≠ OLAP ────────────────────────────────────────
# WHY: Normalized tables are great for INSERT/UPDATE (OLTP).
#      But analytics queries JOIN many tables repeatedly — slow at scale.
#      Star schema PRE-JOINS (denormalizes) for analytical reads (OLAP).
print()
print("=== The Insight ===")
print("Normalized (OLTP) = fast writes, complex reads (JOINs on every query)")
print("Star Schema (OLAP) = designed for reads, JOINs pre-computed at load time")
print()
print("This is what data modeling IS:")
print("  Deciding how to organize data so queries are fast and storage is efficient.")
print("  OLTP = optimize for writes. OLAP (star schema) = optimize for reads.")
print("  Today we build the OLAP layer on top of our existing OLTP call center DB.")


In [ ]:
# =============================================================================
# SECTION A: OLTP SETUP (Online Transaction Processing)
# =============================================================================
# OLTP = the operational database that runs the business in real time.
# It is normalized (3NF = Third Normal Form) — minimal data duplication.
# Each fact lives in exactly ONE place: clients, sources, calls, orders, payments.
# Optimized for fast INSERT/UPDATE of individual rows, not analytical reads.
# =============================================================================

import sqlite3
from datetime import date, timedelta

# In-memory SQLite DB — resets each run; perfect for learning (no cleanup needed)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# --- Create OLTP tables (same schema as M02/M03) ---
# executescript() runs multiple SQL statements separated by semicolons in one call.
# These tables are our SOURCE SYSTEM — the cash register, not the report.
cursor.executescript('''
CREATE TABLE clients (
    client_id INTEGER PRIMARY KEY,  -- PK = Primary Key: uniquely identifies each client row
    client_name TEXT NOT NULL,       -- NOT NULL = required field; no client without a name
    industry TEXT                    -- nullable: some clients may not have industry set
);

CREATE TABLE sources (
    dnis TEXT PRIMARY KEY,           -- DNIS = Dialed Number Identification Service (the phone number dialed)
                                     -- Used as PK because each phone number is a unique campaign entry point
    campaign_name TEXT NOT NULL,
    client_name TEXT NOT NULL,       -- OLTP stores client_name here AND in clients table
                                     -- WHY: This is intentional OLTP denormalization at the source —
                                     --      we'll fix this by normalizing into dim_campaign later
    channel TEXT NOT NULL            -- live_agent or va (Virtual Agent / IVR)
);

CREATE TABLE calls (
    call_id TEXT PRIMARY KEY,        -- PK: unique identifier for each call event
    dnis TEXT NOT NULL,              -- FK pointing to sources.dnis (which campaign was dialed)
    caller_ani TEXT,                 -- ANI = Automatic Number Identification (the caller's phone number)
    call_started_at TEXT NOT NULL,   -- stored as ISO 8601 TEXT in SQLite; real systems use TIMESTAMP type
    call_ended_at TEXT,              -- nullable: a dropped call may have no end time in some systems
    disposition TEXT,                -- outcome: completed | dropped | transferred
    sentiment TEXT,                  -- positive | neutral | negative (from AI sentiment analysis)
    channel TEXT NOT NULL            -- live_agent or va — denormalized here for fast OLTP reads
);

CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    call_id TEXT,                    -- FK: links this order back to the call that generated it
    sku TEXT NOT NULL,               -- SKU = Stock Keeping Unit: the product identifier
    subtotal REAL,
    tax REAL,
    shipping REAL,
    total REAL,
    FOREIGN KEY (call_id) REFERENCES calls(call_id)   -- enforces referential integrity
);

CREATE TABLE payments (
    transaction_id TEXT PRIMARY KEY,
    order_id TEXT,                   -- FK: links payment to the order being paid
    auth_code TEXT,                  -- NULL if declined (no authorization code issued)
    amount REAL,
    status TEXT,                     -- approved | declined
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

CREATE TABLE products (
    sku TEXT PRIMARY KEY,            -- SKU as PK: each product has one canonical record
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    unit_price REAL NOT NULL
);
''')

# =============================================================================
# SECTION B: OLTP DATA LOADING
# =============================================================================
# executemany() inserts multiple rows in one call — more efficient than
# calling execute() in a loop. The ? placeholders prevent SQL injection.
# =============================================================================

# --- Insert OLTP data ---
# 3 clients — the businesses that use our call center platform
cursor.executemany("INSERT INTO clients VALUES (?,?,?)", [
    (1, "Acme Corp", "Retail"),
    (2, "Pinnacle Health", "Healthcare"),
    (3, "Summit Financial", "Financial Services"),
])

# 6 campaigns — each maps one phone number (DNIS) to one client + channel
# WHY 6? Each client has 2 entry points: one live_agent, one va (virtual agent)
cursor.executemany("INSERT INTO sources VALUES (?,?,?,?)", [
    ("8005551234", "Acme Spring Sale", "Acme Corp", "live_agent"),
    ("8005551235", "Acme Rewards Program", "Acme Corp", "va"),
    ("8005552345", "Pinnacle Wellness Plan", "Pinnacle Health", "live_agent"),
    ("8005552346", "Pinnacle Rx Refill", "Pinnacle Health", "va"),
    ("8005553456", "Summit Auto Loan", "Summit Financial", "live_agent"),
    ("8005553457", "Summit Credit Card", "Summit Financial", "va"),
])

# 40 calls — two days of synthetic call center data (March 10 and 11)
# Timestamps are in UTC (note the 'Z' suffix). We'll convert to EST during ETL.
# WHY UTC? Storing in UTC is a best practice — convert to local time at query time,
#          not at write time. Avoids Daylight Saving Time (DST) ambiguity bugs.
cursor.executemany("INSERT INTO calls VALUES (?,?,?,?,?,?,?,?)", [
    ("call_001", "8005551234", "3135559876", "2026-03-10T13:00:00Z", "2026-03-10T13:07:45Z", "completed", "positive", "live_agent"),
    ("call_002", "8005551234", "2485555678", "2026-03-10T13:30:00Z", "2026-03-10T13:32:15Z", "dropped", "negative", "live_agent"),
    ("call_003", "8005552345", "7345551234", "2026-03-10T14:00:00Z", "2026-03-10T14:12:30Z", "completed", "neutral", "live_agent"),
    ("call_004", "8005552346", "3135554321", "2026-03-10T14:30:00Z", "2026-03-10T14:35:00Z", "transferred", "neutral", "va"),
    ("call_005", "8005553456", "2485559999", "2026-03-10T15:00:00Z", "2026-03-10T15:08:20Z", "completed", "positive", "live_agent"),
    ("call_006", "8005551235", "5865551111", "2026-03-10T15:30:00Z", "2026-03-10T15:31:10Z", "dropped", "negative", "va"),
    ("call_007", "8005553457", "3135558888", "2026-03-10T16:00:00Z", "2026-03-10T16:06:45Z", "completed", "positive", "va"),
    ("call_008", "8005551234", "7345557777", "2026-03-10T16:30:00Z", "2026-03-10T16:42:00Z", "completed", "neutral", "live_agent"),
    ("call_009", "8005552345", "2485556666", "2026-03-10T17:00:00Z", "2026-03-10T17:01:30Z", "dropped", "negative", "live_agent"),
    ("call_010", "8005553456", "3135553333", "2026-03-10T17:30:00Z", "2026-03-10T17:38:15Z", "completed", "positive", "live_agent"),
    ("call_011", "8005551234", "5865552222", "2026-03-10T18:00:00Z", "2026-03-10T18:09:30Z", "completed", "neutral", "live_agent"),
    ("call_012", "8005552346", "7345554444", "2026-03-10T18:30:00Z", "2026-03-10T18:33:00Z", "transferred", "neutral", "va"),
    ("call_013", "8005551235", "3135551111", "2026-03-10T19:00:00Z", "2026-03-10T19:05:45Z", "completed", "positive", "va"),
    ("call_014", "8005553457", "2485553333", "2026-03-10T19:30:00Z", "2026-03-10T19:31:00Z", "dropped", "negative", "va"),
    ("call_015", "8005551234", "5865554444", "2026-03-10T20:00:00Z", "2026-03-10T20:11:20Z", "completed", "positive", "live_agent"),
    ("call_016", "8005552345", "7345558888", "2026-03-10T20:30:00Z", "2026-03-10T20:36:00Z", "completed", "neutral", "live_agent"),
    ("call_017", "8005553456", "3135559999", "2026-03-10T21:00:00Z", "2026-03-10T21:02:00Z", "dropped", "negative", "live_agent"),
    ("call_018", "8005551235", "2485551111", "2026-03-10T21:30:00Z", "2026-03-10T21:40:15Z", "completed", "neutral", "va"),
    ("call_019", "8005552346", "5865557777", "2026-03-10T22:00:00Z", "2026-03-10T22:04:30Z", "transferred", "neutral", "va"),
    ("call_020", "8005553457", "7345552222", "2026-03-10T22:30:00Z", "2026-03-10T22:37:45Z", "completed", "positive", "va"),
    ("call_021", "8005551234", "3135556666", "2026-03-11T00:00:00Z", "2026-03-11T00:09:30Z", "completed", "neutral", "live_agent"),
    ("call_022", "8005552345", "2485558888", "2026-03-11T00:30:00Z", "2026-03-11T00:31:45Z", "dropped", "negative", "live_agent"),
    ("call_023", "8005553456", "5865553333", "2026-03-11T01:00:00Z", "2026-03-11T01:06:00Z", "completed", "positive", "live_agent"),
    ("call_024", "8005551235", "7345559999", "2026-03-11T01:30:00Z", "2026-03-11T01:38:20Z", "completed", "positive", "va"),
    ("call_025", "8005553457", "3135552222", "2026-03-11T02:00:00Z", "2026-03-11T02:03:15Z", "transferred", "neutral", "va"),
    ("call_026", "8005551234", "2485554444", "2026-03-11T13:00:00Z", "2026-03-11T13:05:30Z", "completed", "positive", "live_agent"),
    ("call_027", "8005552345", "5865556666", "2026-03-11T13:30:00Z", "2026-03-11T13:32:00Z", "dropped", "negative", "live_agent"),
    ("call_028", "8005553456", "7345553333", "2026-03-11T14:00:00Z", "2026-03-11T14:10:45Z", "completed", "neutral", "live_agent"),
    ("call_029", "8005551235", "3135557777", "2026-03-11T14:30:00Z", "2026-03-11T14:34:00Z", "transferred", "neutral", "va"),
    ("call_030", "8005552346", "2485552222", "2026-03-11T15:00:00Z", "2026-03-11T15:08:30Z", "completed", "positive", "va"),
    ("call_031", "8005553457", "5865559999", "2026-03-11T15:30:00Z", "2026-03-11T15:31:20Z", "dropped", "negative", "va"),
    ("call_032", "8005551234", "7345556666", "2026-03-11T16:00:00Z", "2026-03-11T16:07:00Z", "completed", "neutral", "live_agent"),
    ("call_033", "8005552345", "3135554444", "2026-03-11T16:30:00Z", "2026-03-11T16:41:30Z", "completed", "positive", "live_agent"),
    ("call_034", "8005553456", "2485557777", "2026-03-11T17:00:00Z", "2026-03-11T17:01:45Z", "dropped", "negative", "live_agent"),
    ("call_035", "8005551235", "5865558888", "2026-03-11T17:30:00Z", "2026-03-11T17:36:15Z", "completed", "positive", "va"),
    ("call_036", "8005552346", "7345551111", "2026-03-11T18:00:00Z", "2026-03-11T18:03:30Z", "transferred", "neutral", "va"),
    ("call_037", "8005553457", "3135558888", "2026-03-11T18:30:00Z", "2026-03-11T18:39:45Z", "completed", "neutral", "va"),
    ("call_038", "8005551234", "2485559876", "2026-03-11T19:00:00Z", "2026-03-11T19:06:20Z", "completed", "positive", "live_agent"),
    ("call_039", "8005552345", "5865554321", "2026-03-11T19:30:00Z", "2026-03-11T19:32:00Z", "dropped", "negative", "live_agent"),
    ("call_040", "8005553456", "7345559876", "2026-03-11T20:00:00Z", "2026-03-11T20:07:30Z", "completed", "positive", "live_agent"),
])

# 15 orders — only calls that resulted in a sale (not all 40 calls convert)
# WHY 15 of 40? Roughly 37.5% conversion rate — realistic for a call center
cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?)", [
    ("ord_001", "call_001", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_002", "call_003", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_003", "call_005", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_004", "call_008", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_005", "call_010", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_006", "call_011", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_007", "call_015", "SKU-AC-1003", 89.99, 7.20, 5.80, 102.99),
    ("ord_008", "call_016", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_009", "call_020", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
    ("ord_010", "call_021", "SKU-AC-1001", 69.99, 5.60, 4.40, 79.99),
    ("ord_011", "call_023", "SKU-SF-3001", 39.95, 3.20, 6.80, 49.95),
    ("ord_012", "call_026", "SKU-AC-1002", 139.97, 11.20, 8.80, 159.97),
    ("ord_013", "call_030", "SKU-PH-2001", 99.99, 8.00, 11.99, 119.98),
    ("ord_014", "call_033", "SKU-PH-2002", 59.95, 4.80, 5.20, 69.95),
    ("ord_015", "call_040", "SKU-SF-3002", 149.99, 12.00, 7.00, 168.99),
])

# 15 payments — one per order. Note two "declined" payments (txn_006, txn_012).
# WHY track declined? They still happened — the customer attempted to pay.
#                     is_paid=0 in fact_calls flags these for revenue reconciliation.
cursor.executemany("INSERT INTO payments VALUES (?,?,?,?,?)", [
    ("txn_001", "ord_001", "AUTH-7821", 79.99, "approved"),
    ("txn_002", "ord_002", "AUTH-7822", 119.98, "approved"),
    ("txn_003", "ord_003", "AUTH-7823", 49.95, "approved"),
    ("txn_004", "ord_004", "AUTH-7824", 159.97, "approved"),
    ("txn_005", "ord_005", "AUTH-7825", 49.95, "approved"),
    ("txn_006", "ord_006", None, 79.99, "declined"),        # auth_code is NULL — no approval issued
    ("txn_007", "ord_007", "AUTH-7827", 102.99, "approved"),
    ("txn_008", "ord_008", "AUTH-7828", 69.95, "approved"),
    ("txn_009", "ord_009", "AUTH-7829", 168.99, "approved"),
    ("txn_010", "ord_010", "AUTH-7830", 79.99, "approved"),
    ("txn_011", "ord_011", "AUTH-7831", 49.95, "approved"),
    ("txn_012", "ord_012", None, 159.97, "declined"),       # second declined payment
    ("txn_013", "ord_013", "AUTH-7833", 119.98, "approved"),
    ("txn_014", "ord_014", "AUTH-7834", 69.95, "approved"),
    ("txn_015", "ord_015", "AUTH-7835", 168.99, "approved"),
])

# --- NEW: Products reference table ---
# WHY separate products table? Each product's price and category is stored ONCE.
# If the unit_price changes, we update ONE row — not every order that referenced it.
cursor.executemany("INSERT INTO products VALUES (?,?,?,?)", [
    ("SKU-AC-1001", "Acme Widget Basic", "Widgets", 69.99),
    ("SKU-AC-1002", "Acme Widget Pro Bundle", "Widgets", 139.97),
    ("SKU-AC-1003", "Acme Premium Widget", "Widgets", 89.99),
    ("SKU-PH-2001", "Pinnacle Wellness Kit", "Health & Wellness", 99.99),
    ("SKU-PH-2002", "Pinnacle Supplement Pack", "Health & Wellness", 59.95),
    ("SKU-SF-3001", "Summit Loan Protection Plan", "Financial Products", 39.95),
    ("SKU-SF-3002", "Summit Premium Coverage", "Financial Products", 149.99),
])

conn.commit()

# =============================================================================
# SECTION C: VERIFICATION
# =============================================================================
# Always verify after loading — catch data issues before building on top of them
# =============================================================================

# --- Verify ---
print("=== OLTP Database Rebuilt ===")
for table in ["calls", "orders", "payments", "sources", "clients", "products"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table + ':':<12} {count:>3} rows")
print()
print("This is our OLTP system. Now let's model it for analytics.")
# You should see: calls: 40, orders: 15, payments: 15, sources: 6, clients: 3, products: 7


---

## 2. Star Schema — The Gold Standard

**One-line answer:** A star schema has one fact table (measurements) surrounded by dimension tables (context).
> **Analogy:** Think of a star schema like a hub-and-spoke wheel: `fact_calls` is the hub at the center, and each dimension table (date, time, campaign, product, customer) is a spoke radiating outward. Every measurement (a call, a sale) sits at the hub and connects to context through the spokes.


```
                     dim_date
                        |
  dim_campaign ── fact_calls ── dim_product
                        |
  dim_customer ─────────┘── dim_time
```

| | Fact Table | Dimension Table |
|---|-----------|----------------|
| **Contains** | Measures (numbers) | Context (who, what, when, where) |
| **Rows** | One per event/transaction | One per entity (or version, for SCD) |
| **Changes** | Append-only (new events) | Slowly changes (updates, corrections) |
| **Keys** | Foreign keys to dimensions | Surrogate keys (integers) |
| **Example** | `fact_calls`: duration, revenue, is_converted | `dim_date`: day_name, is_weekend |

**Grain statement:** One row in `fact_calls` = **one phone call**. This is the most important decision in star schema design — everything follows from it.

Our 5 dimensions:
1. **dim_date** — when (day, day of week, weekend flag)
2. **dim_time** — when (hour, AM/PM, time period)
3. **dim_campaign** — which campaign/client (denormalized from sources + clients)
4. **dim_product** — what was sold (from the new products table)
5. **dim_customer** — who called (caller phone number, city, state)

### Surrogate Keys

**Why surrogate keys?** Natural keys (phone numbers, SKUs, campaign names) are strings — slow to join, can change, have gaps. Surrogate keys are auto-incrementing integers: fast, stable, compact.

| Dimension | Natural Key | Surrogate Key |
|-----------|-------------|---------------|
| dim_date | `'2026-03-10'` (TEXT) | `20260310` (INTEGER) |
| dim_time | `'08:00'` (TEXT) | `8` (INTEGER) |
| dim_campaign | `'8005551234'` (TEXT) | `1` (INTEGER) |
| dim_product | `'SKU-AC-1001'` (TEXT) | `1` (INTEGER) |
| dim_customer | `'3135559876'` (TEXT) | `1` (INTEGER) |

**Default dimension row (key = 0):** Every dimension gets a "Unknown/N/A" row with key 0. A call with no order gets `product_key = 0` (Unknown Product) instead of NULL. This eliminates NULLs in the fact table — every foreign key always points to a valid dimension row.

> **Rule:** No NULLs in fact table foreign keys. Use default dimension rows instead.

---

## 3. Building Dimension Tables

**One-line answer:** Dimension tables provide the "who, what, when, where" context that makes fact table numbers meaningful.

Each dimension table below serves a specific purpose:
- **dim_date** — pre-built calendar lookup so analysts filter by day name, weekend, month without parsing timestamps
- **dim_time** — hourly periods (Morning, Afternoon, Evening, Night) for call volume analysis
- **dim_campaign** — campaign details and tier classification
- **dim_customer** — customer info with SCD Type 2 history (Section 4)
- **dim_product** — product catalog for order analysis

In [ ]:
# --- Build dim_date: one row per calendar day ---
# WHY pre-build a date dimension? Two reasons:
#   1. Analysts need human-readable attributes (day_name, is_weekend) — not raw timestamps
#   2. Date math (parsing, formatting) is done ONCE at load time, not on every query
#      In BigQuery, parsing timestamps on 1B rows per query is extremely expensive.
cursor.execute('''
    CREATE TABLE dim_date (
        date_key INTEGER PRIMARY KEY,  -- PK: YYYYMMDD integer — fast to JOIN, readable (e.g., 20260310)
        full_date TEXT NOT NULL,        -- ISO 8601 date string — easy human reading and range filters
        year INTEGER,
        month INTEGER,
        day INTEGER,
        day_of_week INTEGER,            -- 1=Monday ... 7=Sunday (ISO weekday numbering)
        day_name TEXT,                  -- "Monday", "Tuesday" — analysts use this, not the integer
        is_weekend INTEGER              -- 1 if Saturday or Sunday, 0 otherwise (boolean-as-integer in SQLite)
    )
''')

# WHY: Store day_names in a list — avoids calling a function or CASE statement for every date
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
start = date(2026, 3, 1)  # March 2026 — our call center data covers this month

# Build one row per day for the entire month (31 days)
# BEGINNER NOTE: timedelta(days=i) adds i days to a date — Python's way to do date arithmetic
for i in range(31):
    d = start + timedelta(days=i)
    cursor.execute(
        "INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?)",
        (int(d.strftime('%Y%m%d')), d.isoformat(), d.year, d.month, d.day,
         d.isoweekday(), day_names[d.weekday()], 1 if d.weekday() >= 5 else 0)
        # d.isoweekday() → 1-7 (Mon-Sun); d.weekday() → 0-6 (Mon-Sun)
        # weekday() >= 5 means Saturday (5) or Sunday (6) → is_weekend = 1
    )

conn.commit()

# --- Display ---
print("=== dim_date (March 2026) ===")
print(f"{'date_key':>10}  {'full_date':<12}  {'year':>4}  {'month':>5}  {'day':>3}  {'dow':>3}  {'day_name':<11}  {'weekend':>7}")
print("-" * 72)
rows = cursor.execute("SELECT * FROM dim_date LIMIT 7").fetchall()
for r in rows:
    wknd = "Yes" if r[7] else "No"
    print(f"{r[0]:>10}  {r[1]:<12}  {r[2]:>4}  {r[3]:>5}  {r[4]:>3}  {r[5]:>3}  {r[6]:<11}  {wknd:>7}")

total = cursor.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0]
print(f"\n{total} rows total — one per calendar day.")
print("Analysts filter by day_name, is_weekend — they never parse timestamps.")
# You should see: 31 rows for March 2026, Monday=1 through Sunday=7


In [ ]:
# --- Build dim_time: one row per hour (0-23) ---
# WHY a separate time dimension? Call centers care deeply about WHEN calls happen.
# "Morning vs Evening conversion rate" is a critical operational metric.
# Pre-bucketing into time_period (Morning/Afternoon/Evening/Night) means
# analysts never need to write CASE WHEN hour < 12 THEN 'Morning' ... in every query.
# That logic lives in one place: this dimension.
cursor.execute('''
    CREATE TABLE dim_time (
        time_key INTEGER PRIMARY KEY,  -- PK: the hour (0-23) doubles as the natural key
        hour_24 INTEGER NOT NULL,       -- 24-hour clock (0-23) — used in JOINs from fact table
        hour_12 INTEGER NOT NULL,       -- 12-hour clock (1-12) — for display in dashboards
        am_pm TEXT NOT NULL,            -- "AM" or "PM" — human-friendly display
        time_period TEXT NOT NULL       -- buckets: Night / Morning / Afternoon / Evening
    )
''')

# WHY these time bucket boundaries?
#   Night (0-5):      Most call centers are closed or have minimal staffing
#   Morning (6-11):   Business opens, first wave of customers
#   Afternoon (12-17): Lunch through end of business — typically peak volume
#   Evening (18-23):  After-work callers; often second peak for retail/health
for h in range(24):
    hour_12 = h % 12 if h % 12 != 0 else 12  # convert 0→12, 13→1, etc.
    am_pm = 'AM' if h < 12 else 'PM'
    if h < 6:
        period = 'Night'
    elif h < 12:
        period = 'Morning'
    elif h < 18:
        period = 'Afternoon'
    else:
        period = 'Evening'
    cursor.execute("INSERT INTO dim_time VALUES (?,?,?,?,?)", (h, h, hour_12, am_pm, period))

conn.commit()

# --- Display all 24 rows ---
print("=== dim_time (24 hours) ===")
print(f"{'time_key':>8}  {'hour_24':>7}  {'hour_12':>7}  {'am_pm':<5}  {'time_period':<10}")
print("-" * 44)
for r in cursor.execute("SELECT * FROM dim_time"):
    print(f"{r[0]:>8}  {r[1]:>7}  {r[2]:>7}  {r[3]:<5}  {r[4]:<10}")

print("\ntime_period lets analysts ask 'How do evening calls compare to morning?'")
print("without knowing the hour boundaries.")
# You should see: 24 rows, hour 0 = Night/12AM, hour 6 = Morning, hour 12 = Afternoon, hour 18 = Evening


In [ ]:
# --- Build dim_campaign: denormalize sources + clients ---
# WHY denormalize here? In OLTP, sources and clients are separate tables (normalized).
# Every OLTP query joining sources + clients costs a JOIN at read time.
# In the star schema, we flatten them into ONE table at ETL load time.
# Result: analysts get campaign + client + industry in a single JOIN — not three.
# This is intentional OLAP denormalization: trade storage for query speed.
cursor.execute('''
    CREATE TABLE dim_campaign (
        campaign_key INTEGER PRIMARY KEY AUTOINCREMENT,  -- AUTOINCREMENT: database generates the next integer automatically — no manual ID management
        dnis TEXT NOT NULL,              -- DNIS kept for traceability back to OLTP source
        campaign_name TEXT NOT NULL,
        client_name TEXT NOT NULL,
        industry TEXT,
        channel TEXT NOT NULL            -- live_agent or va — denormalized from sources
    )
''')

# Flatten the JOIN — one table instead of two
# WHY: This INSERT…SELECT does the denormalization at load time (ETL).
#      After this, the star schema never needs to JOIN sources + clients again.
cursor.execute('''
    INSERT INTO dim_campaign (dnis, campaign_name, client_name, industry, channel)
    SELECT s.dnis, s.campaign_name, s.client_name, cl.industry, s.channel
    FROM sources s
    JOIN clients cl ON s.client_name = cl.client_name  -- OLTP JOIN done once here at load time
    ORDER BY s.dnis                                      -- deterministic order for consistent surrogate keys
''')

conn.commit()

# --- Display ---
print("=== dim_campaign ===")
print(f"{'key':>3}  {'dnis':<12}  {'campaign_name':<24}  {'client_name':<18}  {'industry':<20}  {'channel':<10}")
print("-" * 92)
for r in cursor.execute("SELECT * FROM dim_campaign"):
    print(f"{r[0]:>3}  {r[1]:<12}  {r[2]:<24}  {r[3]:<18}  {r[4]:<20}  {r[5]:<10}")

print("\nIn OLTP, sources and clients are separate tables (normalized).")
print("In the star schema, we denormalize — one table, no JOINs needed.")
# You should see: 6 rows, campaign_key 1-6, each with industry filled in from clients table


In [ ]:
# --- Build dim_product: products + default row ---
# WHY a separate dim_product? In OLTP, the products table is already normalized.
# We simply load it into the star schema as a dimension — with one critical addition:
# the DEFAULT ROW (key=0). Not every call results in a sale, so many fact rows
# will have no matching product. Instead of storing NULL in fact_calls.product_key,
# we point those rows to the "Unknown Product" default row.
# BEGINNER NOTE: NULLs in fact table FK columns break many BI tools and analytics queries.
#               The default row is an industry-standard solution — always add it.
cursor.execute('''
    CREATE TABLE dim_product (
        product_key INTEGER PRIMARY KEY,  -- PK: manually assigned integers (not AUTOINCREMENT)
        sku TEXT,                          -- SKU = Stock Keeping Unit (product identifier in OLTP)
                                           -- nullable on the default row (row 0 has no real SKU)
        product_name TEXT NOT NULL,
        category TEXT NOT NULL,
        unit_price REAL                    -- nullable on default row; 0.00 for "Unknown Product"
    )
''')

# Row 0 = default for calls with no order
# WHY 0? Convention: surrogate key 0 = "not applicable" / "unknown"
#         Real data starts at key=1. This makes it easy to filter: WHERE product_key > 0
products_dim = [
    (0, "N/A", "Unknown Product", "Unknown", 0.00),  # default row — no sale on this call
    (1, "SKU-AC-1001", "Acme Widget Basic", "Widgets", 69.99),
    (2, "SKU-AC-1002", "Acme Widget Pro Bundle", "Widgets", 139.97),
    (3, "SKU-AC-1003", "Acme Premium Widget", "Widgets", 89.99),
    (4, "SKU-PH-2001", "Pinnacle Wellness Kit", "Health & Wellness", 99.99),
    (5, "SKU-PH-2002", "Pinnacle Supplement Pack", "Health & Wellness", 59.95),
    (6, "SKU-SF-3001", "Summit Loan Protection Plan", "Financial Products", 39.95),
    (7, "SKU-SF-3002", "Summit Premium Coverage", "Financial Products", 149.99),
]

cursor.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?)", products_dim)
conn.commit()

# --- Display ---
print("=== dim_product ===")
print(f"{'key':>3}  {'sku':<14}  {'product_name':<30}  {'category':<20}  {'price':>8}")
print("-" * 80)
for r in cursor.execute("SELECT * FROM dim_product"):
    print(f"{r[0]:>3}  {r[1]:<14}  {r[2]:<30}  {r[3]:<20}  ${r[4]:>7.2f}")

print("\n8 rows (7 products + 1 default).")
print("The default row (key=0) avoids NULLs in fact_calls. Every call has a product_key.")
# You should see: row 0 = Unknown Product, rows 1-7 = real products across 3 categories


---

## 4. Slowly Changing Dimensions (SCD)

**One-line answer:** SCD Type 2 preserves history by adding a new row when an attribute changes, instead of overwriting.
> **Analogy:** Think of SCD Type 2 like passport history. When you renew your passport, the government doesn’t erase the old one — they keep both on file with date ranges. If a crime happened in 2019, they check which passport was valid then, not the one you hold today. That’s exactly how Type 2 works: every version of a customer record is kept, with `effective_date` and `end_date` marking when each version was active.


| Type | Strategy | History? | Example |
|------|----------|----------|---------|
| **Type 0** | Never update | N/A | Date of birth — immutable |
| **Type 1** | Overwrite the value | Lost | Fix a typo in a name — old value gone |
| **Type 2** | Add new row with date range | Preserved | Customer moves cities — both addresses kept |
| **Type 3** | Add column for previous value | Limited | `current_city` + `previous_city` — only 1 version back |

> **Type 2 is the most important for Data Engineers.** It answers the question: *"What was true at the time of the transaction?"*

### How SCD Type 2 Works

Each row gets three extra columns:
- `effective_date` — when this version became active
- `end_date` — when this version was replaced (9999-12-31 = still active)
> **Why `9999-12-31`?** The date `9999-12-31` is an industry convention meaning “still active” — it’s easier to query than NULL because you can write `WHERE end_date > '2024-01-15'` without special NULL handling.

- `is_current` — 1 for the active row, 0 for historical rows

When a customer moves from Detroit to Chicago on March 11:
1. **Close** the old row: set `end_date = '2026-03-10'`, `is_current = 0`
2. **Insert** a new row: Chicago, `effective_date = '2026-03-11'`, `end_date = '9999-12-31'`, `is_current = 1`

In [ ]:
# --- Build dim_customer with SCD Type 2 ---
# WHY SCD Type 2 here specifically? Customer location changes over time.
# Business question: "What city was the customer in WHEN they called?"
# Not "what city are they in NOW?" — those are different questions.
# SCD Type 2 (Slowly Changing Dimension Type 2) preserves the full history
# by adding a NEW ROW each time an attribute changes. The old row stays,
# with end_date stamped to show when it became inactive.
cursor.execute('''
    CREATE TABLE dim_customer (
        customer_key INTEGER PRIMARY KEY,  -- PK: surrogate key (integer), not the phone number
        caller_ani TEXT NOT NULL,           -- ANI = Automatic Number Identification (caller's phone)
                                            -- This is the natural key — but NOT the PK.
                                            -- WHY? Because the same caller_ani can appear MULTIPLE times
                                            --      (once per version of their record in SCD Type 2).
                                            --      A natural key must be unique; here it is not.
        city TEXT,
        state TEXT,
        effective_date TEXT NOT NULL,       -- when this version became active (INCLUSIVE)
        end_date TEXT NOT NULL,             -- when this version was replaced (INCLUSIVE)
                                            -- WHY '9999-12-31' instead of NULL for "still active"?
                                            -- BEGINNER NOTE: '9999-12-31' lets you write
                                            --   WHERE call_date BETWEEN effective_date AND end_date
                                            -- If end_date were NULL, BETWEEN would fail (NULL comparisons
                                            -- always return NULL, not TRUE). Industry-standard convention.
        is_current INTEGER NOT NULL DEFAULT 1  -- 1 = this is the active version, 0 = historical
    )
''')

# Area code to city/state mapping (Michigan area codes for our synthetic callers)
# WHY Michigan? Our fictional call center is based in Michigan — consistent with M02/M03
area_city = {
    '313': ('Detroit', 'MI'),
    '248': ('Southfield', 'MI'),
    '734': ('Ann Arbor', 'MI'),
    '586': ('Warren', 'MI'),
}

# Extract unique callers and assign surrogate keys
# enumerate(callers, start=1) assigns surrogate keys 1, 2, 3, ... — 0 is reserved for default
callers = cursor.execute(
    "SELECT DISTINCT caller_ani FROM calls ORDER BY caller_ani"
).fetchall()

for i, (ani,) in enumerate(callers, start=1):
    area_code = ani[:3]  # first 3 digits = area code
    city, state = area_city.get(area_code, ('Unknown', 'XX'))  # default if area code not in map
    cursor.execute(
        "INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?)",
        (i, ani, city, state, '2026-03-01', '9999-12-31', 1)
        # effective_date = start of month (when we first saw this customer in our system)
        # end_date = '9999-12-31' = "currently active" — the industry convention for open-ended rows
    )

# --- SCD Type 2 Demo ---
# Customer 3135559876 moved from Detroit, MI to Chicago, IL on March 11
# This is the SCD Type 2 update pattern — TWO steps, always:

# Step 1: Close the current row (stamp the end date, flip is_current to 0)
# WHY end_date = '2026-03-10'? The last day this version was valid was March 10.
#     On March 11, the new version takes over.
cursor.execute('''
    UPDATE dim_customer
    SET end_date = '2026-03-10', is_current = 0
    WHERE caller_ani = '3135559876' AND is_current = 1
''')

# Step 2: Insert new row with updated city (the new version)
# WHY a new row and not an UPDATE? Because we NEVER overwrite in SCD Type 2.
# If we updated in place, the March 10 call would now incorrectly say "Chicago."
next_key = len(callers) + 1  # one beyond the last assigned surrogate key
cursor.execute(
    "INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?)",
    (next_key, '3135559876', 'Chicago', 'IL', '2026-03-11', '9999-12-31', 1)
)

conn.commit()

# --- Display ---
total = cursor.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"=== dim_customer (SCD Type 2) — {total} rows ===")
print()
print(f"{'key':>4}  {'caller_ani':<12}  {'city':<12}  {'st':<3}  {'effective':<12}  {'end_date':<12}  {'cur':>3}")
print("-" * 68)
rows = cursor.execute("SELECT * FROM dim_customer LIMIT 5").fetchall()
for r in rows:
    print(f"{r[0]:>4}  {r[1]:<12}  {r[2]:<12}  {r[3]:<3}  {r[4]:<12}  {r[5]:<12}  {r[6]:>3}")
print("...")

# Show the SCD2 rows — same ANI, two rows, different date ranges
print()
print("=== SCD Type 2 Demo: Caller 3135559876 ===")
scd_rows = cursor.execute(
    "SELECT * FROM dim_customer WHERE caller_ani = '3135559876' ORDER BY effective_date"
).fetchall()
for r in scd_rows:
    print(f"{r[0]:>4}  {r[1]:<12}  {r[2]:<12}  {r[3]:<3}  {r[4]:<12}  {r[5]:<12}  {r[6]:>3}")
print()
print("Same person, two rows. The effective_date tells you which row was active when.")
# You should see: same caller_ani in two rows — row 1 ends 2026-03-10, row 2 starts 2026-03-11


In [ ]:
# --- Point-in-Time Lookup: SCD Type 2 ---
# This is the core SCD Type 2 query pattern — "what was true on a given date?"
# The key SQL trick: BETWEEN effective_date AND end_date
# WHY BETWEEN? It returns the ONE row whose date range contains the target date.
#              Since effective and end dates don't overlap across versions,
#              exactly one row will match for any given date.
# BEGINNER NOTE: This is why end_date = '9999-12-31' matters — if it were NULL,
#                the BETWEEN comparison would return NULL (not TRUE), and no row
#                would match for the currently-active version.

# Where was caller 3135559876 on March 10?
print("=== Where was 3135559876 on March 10? ===")
row = cursor.execute('''
    SELECT customer_key, caller_ani, city, state, effective_date, end_date
    FROM dim_customer
    WHERE caller_ani = '3135559876'
      AND '2026-03-10' BETWEEN effective_date AND end_date
''').fetchone()
print(f"  Key: {row[0]}, City: {row[2]}, State: {row[3]}")
print(f"  Active: {row[4]} to {row[5]}")

# Where was caller 3135559876 on March 12?
# WHY same query, different answer? Because a different row's date range contains March 12.
print()
print("=== Where was 3135559876 on March 12? ===")
row = cursor.execute('''
    SELECT customer_key, caller_ani, city, state, effective_date, end_date
    FROM dim_customer
    WHERE caller_ani = '3135559876'
      AND '2026-03-12' BETWEEN effective_date AND end_date
''').fetchone()
print(f"  Key: {row[0]}, City: {row[2]}, State: {row[3]}")
print(f"  Active: {row[4]} to {row[5]}")

print()
print("Same person, different answer depending on WHEN you ask.")
print("This is how you answer 'Where did the customer live WHEN they called?'")
print("Not where they live NOW.")
# You should see: March 10 → Detroit, MI (key=1). March 12 → Chicago, IL (key=21 or similar).


## Why Track History? What Business Decisions Require Knowing the OLD Value?

SCD Type 2 exists because "what is true now" is not the same as "what was true then."

**Call center example — the one we just built:**
- Caller 3135559876 was in Detroit on March 10 and Chicago on March 11
- On March 10 they bought a Summit Loan Protection Plan
- If their city now shows "Chicago" for that sale, the regional revenue report is wrong
- Detroit's regional manager loses credit. Chicago's gets credit they didn't earn.

**Real business scenarios where SCD Type 2 matters:**

| Business Question | Why History Required |
|---|---|
| "Which region had the highest revenue last quarter?" | Customer may have moved since then |
| "What was the price of this product when it was ordered?" | Product prices change — orders must lock in the price at sale time |
| "Which sales rep managed this account when the deal closed?" | Reps change territories and companies |
| "What was the customer's tier when they called?" | Gold → Silver downgrades shouldn't change historical reporting |
| "Which version of our campaign ran during the spike?" | Campaign names, targeting, and messaging change over time |

**The audit / regulatory angle:**
In healthcare and financial services, you are often **legally required** to prove what was true at a specific point in time. "The patient was covered under Plan A when this procedure was performed" — that's an SCD Type 2 lookup. Overwriting history (SCD Type 1) would destroy that evidence.

**Interview answer:** *"SCD Type 2 preserves history by inserting a new row with a date range when an attribute changes. The fact table captures the surrogate key of the version active at event time, so point-in-time queries remain accurate even after the source data changes."*


## SCD Type 1 vs. Type 2 vs. Type 3 — When to Use Which?

| Type | Action | History? | Storage cost | Best for |
|---|---|---|---|---|
| **Type 0** | Never update (freeze) | N/A | None | Immutable facts (date of birth, original signup date) |
| **Type 1** | Overwrite in place | Lost forever | None | Correcting data errors, non-analytical attributes |
| **Type 2** | Add new row with date range | Full history | Grows with changes | Any attribute used in time-sensitive analytics |
| **Type 3** | Add a "previous value" column | Only 1 version back | Fixed overhead | "Current vs. previous" reports (rare; avoid in most cases) |

**Decision guide:**
- Ask "Do I ever need to ask 'what was true THEN?'"
  - YES → Type 2
  - NO → Type 1
- Is this an error (wrong value entered)? → Type 1 (overwrite the mistake)
- Is this a real business change (customer moved, product repriced)? → Type 2 (preserve both versions)

**Type 3 red flag:** If you find yourself wanting to track more than "current" and "previous," you need Type 2. Type 3 caps you at one version back.

---

### Interview Questions: SCD

**Q1:** *"A customer's email address was entered incorrectly at signup. How do you fix it in the dimension?"*
A: SCD Type 1 — it's an error, not a real change. Overwrite the bad email. No history needed.

**Q2:** *"A customer moves from Detroit to Chicago. How do you handle this?"*
A: SCD Type 2 — it's a real change that affects geographic analysis. New row, date-range the old one.

**Q3:** *"The fact table has 200 million rows. A customer SCD update will create 1 million new dim_customer rows. How do you handle the fact table's historical foreign keys?"*
A: Nothing — the fact table's historical rows already point to the OLD surrogate key (the Detroit version). New fact rows will point to the new Chicago surrogate key. The fact table doesn't need updating. That's the entire point of surrogate keys in SCD Type 2.

**Q4:** *"What's the danger of using the natural key (caller_ani) as the fact table FK instead of the surrogate key?"*
A: A natural key can't distinguish between the Detroit version and the Chicago version of the same caller — it's the same phone number. The surrogate key is what makes SCD Type 2 work.


---

## 5. Building the Fact Table

**Grain:** One row in `fact_calls` = one phone call.

**Foreign keys** (pointers to dimensions):
- `date_key` → dim_date
- `time_key` → dim_time
- `campaign_key` → dim_campaign
- `product_key` → dim_product (0 if no order)
- `customer_key` → dim_customer (SCD Type 2 date-range lookup)

**Measures** (numbers for aggregation):
- `duration_seconds`, `duration_minutes` — call length
- `order_amount` — revenue (0 if no order)
- `is_converted` — 1 if the call generated an order, 0 otherwise
- `is_paid` — 1 if payment was approved, 0 otherwise

**Degenerate dimensions** (a dimension value stored directly in the fact table instead of in a separate dimension table — like `call_id`, which is unique per row and doesn’t need a lookup):
- `disposition` — completed, dropped, transferred
- `sentiment` — positive, neutral, negative

> **Rule:** The fact table has numbers (measures) and keys (pointers). No descriptive text — except degenerate dimensions that don't warrant their own table.

In [ ]:
# --- Build fact_calls ---
# The fact table is the CENTER of the star schema.
# WHY is it in the center? Because every row captures ONE BUSINESS EVENT (a call),
# and every measurement (duration, revenue, conversion) belongs to that event.
# Dimension tables provide context; the fact table holds the numbers.
# GRAIN STATEMENT: One row in fact_calls = one phone call. This is locked.
cursor.execute('''
    CREATE TABLE fact_calls (
        call_id TEXT PRIMARY KEY,          -- DEGENERATE DIMENSION: the OLTP call ID stored directly
                                            -- in the fact table. "Degenerate" = it doesn't have
                                            -- its own dimension table (call_id is unique per row).
        date_key INTEGER NOT NULL,          -- FK → dim_date: the date portion of call_started_at (EST)
        time_key INTEGER NOT NULL,          -- FK → dim_time: the hour (0-23) of call_started_at (EST)
        campaign_key INTEGER NOT NULL,      -- FK → dim_campaign: which phone number / campaign was dialed
        product_key INTEGER NOT NULL,       -- FK → dim_product: what was ordered (0 = no order)
        customer_key INTEGER NOT NULL,      -- FK → dim_customer: the SCD Type 2 version active at call time
        duration_seconds INTEGER,           -- MEASURE: call length in seconds (for precise math)
        duration_minutes REAL,              -- MEASURE: call length in minutes (rounded, for reporting)
        order_amount REAL DEFAULT 0,        -- MEASURE: revenue from this call's order (0 if no order)
        disposition TEXT,                   -- DEGENERATE DIMENSION: completed | dropped | transferred
        sentiment TEXT,                     -- DEGENERATE DIMENSION: positive | neutral | negative
        is_converted INTEGER DEFAULT 0,     -- MEASURE (boolean flag): 1 if call produced an order, 0 if not
        is_paid INTEGER DEFAULT 0,          -- MEASURE (boolean flag): 1 if payment was approved, 0 if not
        FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY (time_key) REFERENCES dim_time(time_key),
        FOREIGN KEY (campaign_key) REFERENCES dim_campaign(campaign_key),
        FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
        FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key)
    )
''')

# Transform OLTP → fact table with dimension key lookups
# KEY SQL CONCEPTS in this query:
# - julianday() converts timestamps to decimal days; subtract two = elapsed time
#   * 86400 converts days to seconds for duration
# - COALESCE(..., 0) returns 0 when no matching product/order exists (instead of NULL)
# - '-5 hours': converts UTC to EST (fixes the UTC bug from M02)
# - The SCD Type 2 JOIN: match the customer version that was active at call time
cursor.execute('''
    INSERT INTO fact_calls
    SELECT
        c.call_id,
        dd.date_key,
        dt.time_key,
        dcamp.campaign_key,
        -- COALESCE(a, b) = returns a if not NULL, otherwise returns b. Prevents NULL contamination.
        -- WHY: Some calls have no order → dprod.product_key would be NULL → use 0 (default row).
        COALESCE(dprod.product_key, 0) AS product_key,
        dcust.customer_key,
        -- julianday() = SQLite's way to do date math. Returns days as a decimal number.
        -- Multiply by 86400 (seconds per day) to get duration in seconds.
        CAST((julianday(c.call_ended_at) - julianday(c.call_started_at)) * 86400 AS INTEGER),
        -- Multiply by 1440 (minutes per day) for duration in minutes, rounded to 2 decimal places.
        ROUND((julianday(c.call_ended_at) - julianday(c.call_started_at)) * 1440, 2),
        COALESCE(o.total, 0.0),            -- 0.0 if no order (LEFT JOIN returns NULL for unmatched calls)
        c.disposition,
        c.sentiment,
        -- CASE WHEN: boolean flags — cleaner than storing NULL for "no order"
        CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END,
        CASE WHEN pay.status = 'approved' THEN 1 ELSE 0 END
    FROM calls c
    JOIN dim_date dd
        ON dd.full_date = DATE(c.call_started_at, '-5 hours')
        -- WHY '-5 hours'? Timestamps stored in UTC. EST = UTC-5.
        --   A call at 2026-03-10T01:00:00Z is 2026-03-09 8PM EST — a different date!
        --   Convert once here in the ETL, never again in analytics queries.
    JOIN dim_time dt
        ON dt.time_key = CAST(strftime('%H', c.call_started_at, '-5 hours') AS INTEGER)
        -- strftime('%H', ...) extracts the hour (00-23) as a string; CAST converts to integer
    JOIN dim_campaign dcamp
        ON dcamp.dnis = c.dnis            -- match the phone number dialed to the campaign dimension
    JOIN dim_customer dcust
        ON dcust.caller_ani = c.caller_ani
        AND DATE(c.call_started_at, '-5 hours') BETWEEN dcust.effective_date AND dcust.end_date
        -- WHY this AND clause? This is the SCD Type 2 point-in-time JOIN.
        --   Without it, callers with 2 SCD rows would return 2 fact rows (a fan-out / data explosion).
        --   The BETWEEN ensures we pick the ONE customer version active at call time.
    LEFT JOIN orders o
        ON o.call_id = c.call_id          -- LEFT JOIN: keep all calls, even those with no order
    LEFT JOIN payments pay
        ON pay.order_id = o.order_id      -- LEFT JOIN: keep all orders, even those with no payment
    LEFT JOIN dim_product dprod
        ON dprod.sku = o.sku              -- LEFT JOIN: keep all calls, COALESCE handles the NULL above
''')

conn.commit()

# --- Display ---
total = cursor.execute("SELECT COUNT(*) FROM fact_calls").fetchone()[0]
print(f"=== fact_calls — {total} rows ===")
print()
print(f"{'call_id':<10} {'date_key':>8} {'time':>4} {'camp':>4} {'prod':>4} {'cust':>4}  {'dur_s':>5}  {'amount':>8}  {'disp':<11} {'conv':>4} {'paid':>4}")
print("-" * 88)
rows = cursor.execute("SELECT * FROM fact_calls LIMIT 10").fetchall()
for r in rows:
    print(f"{r[0]:<10} {r[1]:>8} {r[2]:>4} {r[3]:>4} {r[4]:>4} {r[5]:>4}  {r[6]:>5}  ${r[8]:>7.2f}  {r[9]:<11} {r[11]:>4} {r[12]:>4}")
print("... (showing first 10)")

# Verify counts
converted = cursor.execute("SELECT SUM(is_converted) FROM fact_calls").fetchone()[0]
paid = cursor.execute("SELECT SUM(is_paid) FROM fact_calls").fetchone()[0]
revenue = cursor.execute("SELECT SUM(order_amount) FROM fact_calls").fetchone()[0]
print(f"\n40 OLTP calls -> {total} fact rows. {converted} conversions, {paid} paid, ${revenue:,.2f} total revenue.")
print("Each row points to 5 dimensions via integer keys.")
# You should see: 40 rows, 15 conversions, 13 paid, total revenue ~$1,378 (2 declined)


## Why Is fact_calls the CENTER of the Star? What Makes Something a Fact vs. a Dimension?

**The test for "is this a fact or a dimension?"**

Ask yourself two questions:
1. **Can you aggregate it?** (SUM, AVG, COUNT) — If yes, it's probably a **measure** (lives in the fact table).
2. **Does it describe context — who, what, when, where?** — If yes, it's probably a **dimension**.

**Applying the test to our schema:**

| Column | Fact or Dimension? | Why |
|---|---|---|
| `duration_seconds` | Fact (measure) | You SUM it: "total call minutes this week" |
| `order_amount` | Fact (measure) | You SUM it: "total revenue by campaign" |
| `is_converted` | Fact (measure/flag) | You SUM it: "conversion rate = SUM/COUNT" |
| `day_name` | Dimension | Describes WHEN — you GROUP BY it, not aggregate it |
| `campaign_name` | Dimension | Describes WHICH campaign — context, not a number |
| `city` | Dimension | Describes WHERE the customer called from |
| `disposition` | Degenerate dimension | Text label unique per call — small enough to store in fact table |

**Why the fact table is the CENTER:**
Every business question starts with a measurement: "How much? How many? How long?"
Those measurements live in the fact table. Dimensions answer the follow-up: "...by what? ...for whom? ...during when?"

The star structure puts measurements at the center because that's how questions are actually asked:
- "What is our **revenue** [fact] by **campaign** [dimension] by **day of week** [dimension]?"
- "What is our **conversion rate** [fact] by **time period** [dimension] by **client** [dimension]?"

**The grain statement is everything:**
Once you say "one row = one phone call," everything else follows.
Duration, revenue, and conversion belong to a call. Date, campaign, and customer are the context of a call.
If your grain were "one row = one order," the schema would look different — duration wouldn't belong, but quantity and unit_price would.


---

## 6. Star Schema Queries

**Now the payoff.** Queries on a star schema are simpler, faster, and more intuitive than OLTP joins. No timestamp parsing, no multi-table chains, no string matching — just clean dimension filters.

In [ ]:
# --- Query 1: Revenue by Day of Week ---
# WHY start here? Day of week is the most common first-cut analysis in call centers.
# "Do we get more revenue on Tuesdays than Mondays?" — one JOIN, done.
# In OLTP, this would require timestamp parsing on every query. Here it's a simple JOIN.
# Display the results so you can verify the output matches expectations
print("=== Revenue by Day of Week ===")
print(f"{'day_name':<12} {'calls':>5} {'orders':>6} {'revenue':>10}")
print("-" * 36)
for r in cursor.execute('''
    SELECT dd.day_name, COUNT(*) AS calls,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_date dd ON f.date_key = dd.date_key  -- single JOIN replaces timestamp parsing
    GROUP BY dd.day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>6}  ${r[3]:>8.2f}")

# --- Query 2: Conversion Rate by Time Period ---
# WHY conversion rate? Because raw call volume doesn't tell you if the calls are profitable.
# Conversion rate = orders / calls — the key performance metric for a call center.
print()
print("=== Conversion Rate by Time Period ===")
print(f"{'time_period':<12} {'calls':>5} {'conv':>5} {'rate':>7}")
print("-" * 32)
for r in cursor.execute('''
    SELECT dt.time_period, COUNT(*) AS calls,
           SUM(f.is_converted) AS conversions,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_time dt ON f.time_key = dt.time_key  -- time_period comes pre-bucketed from dim_time
    GROUP BY dt.time_period
    ORDER BY conv_rate DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>5} {r[3]:>6.1f}%")

# --- Query 3: Revenue by Product Category ---
# WHY filter WHERE product_key > 0? Row 0 = "Unknown Product" (calls with no order).
# Including it would show $0 revenue for "Unknown" — misleading on a revenue report.
# Filtering to product_key > 0 means "only calls that resulted in a sale."
print()
print("=== Revenue by Product Category ===")
print(f"{'category':<22} {'orders':>6} {'revenue':>10}")
print("-" * 40)
for r in cursor.execute('''
    SELECT dp.category,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_product dp ON f.product_key = dp.product_key
    WHERE dp.product_key > 0  -- exclude the default "Unknown Product" row (key=0)
    GROUP BY dp.category
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<22} {r[1]:>6}  ${r[2]:>8.2f}")

# --- Query 4: Campaign Performance Dashboard ---
# WHY this last? It's the most business-critical view — which campaigns earn more per call?
# conv_rate + revenue combined tells you both volume AND quality of each campaign.
print()
print("=== Campaign Performance Dashboard ===")
print(f"{'campaign':<24} {'client':<18} {'calls':>5} {'orders':>6} {'revenue':>10} {'conv%':>6}")
print("-" * 74)
for r in cursor.execute('''
    SELECT dcamp.campaign_name, dcamp.client_name,
           COUNT(*) AS calls,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_campaign dcamp ON f.campaign_key = dcamp.campaign_key
    GROUP BY dcamp.campaign_name, dcamp.client_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<18} {r[2]:>5} {r[3]:>6}  ${r[4]:>8.2f} {r[5]:>5.1f}%")

print()
print("No complex string parsing, no timestamp math, no multi-table chains.")
print("The star schema pre-computed all of that.")
# You should see: all 6 campaigns ranked by revenue. Lower conv% but higher order_amount = still high revenue.


In [ ]:
# --- Same Question, Two Ways ---
# "Revenue by campaign by day of week (orders only)"
# This is the "before and after" proof that star schema queries are simpler.
# Both queries return IDENTICAL results — but the code complexity is very different.

# OLTP version: 3 JOINs, CASE/strftime for day name, string-based GROUP BY
# WHY so complex? Because OLTP stores raw UTC timestamps — every query must:
#   1. Parse the timestamp (strftime)
#   2. Convert timezone (datetime(..., '-5 hours'))
#   3. Map the weekday number to a day name (CASE WHEN '0' THEN 'Sunday'...)
#   4. JOIN multiple tables to get campaign context
print("=== OLTP Query: Revenue by Campaign by Day ===")
print(f"{'campaign':<24} {'day':<11} {'orders':>6} {'revenue':>10}")
print("-" * 54)
for r in cursor.execute('''
    SELECT s.campaign_name,
           CASE strftime('%w', datetime(c.call_started_at, '-5 hours'))
               WHEN '0' THEN 'Sunday'    WHEN '1' THEN 'Monday'
               WHEN '2' THEN 'Tuesday'   WHEN '3' THEN 'Wednesday'
               WHEN '4' THEN 'Thursday'  WHEN '5' THEN 'Friday'
               WHEN '6' THEN 'Saturday'
           END AS day_name,
           COUNT(*) AS orders,
           SUM(o.total) AS revenue
    FROM calls c
    JOIN sources s ON c.dnis = s.dnis
    JOIN orders o ON o.call_id = c.call_id
    GROUP BY s.campaign_name, day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<11} {r[2]:>6}  ${r[3]:>8.2f}")

# Star schema version: 2 JOINs, clean columns, no parsing
# WHY simpler? Because the ETL (building fact_calls) handled all the complex logic ONCE.
# Now every query reaps the benefit — clean columns, simple JOINs, no functions to call.
print()
print("=== Star Schema Query: Same Question ===")
print(f"{'campaign':<24} {'day':<11} {'orders':>6} {'revenue':>10}")
print("-" * 54)
for r in cursor.execute('''
    SELECT dc.campaign_name, dd.day_name,
           COUNT(*) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_campaign dc ON f.campaign_key = dc.campaign_key  -- campaign name pre-joined
    JOIN dim_date dd ON f.date_key = dd.date_key              -- day_name pre-computed
    WHERE f.is_converted = 1   -- filter to converted calls only (no CASE/IS NOT NULL needed)
    GROUP BY dc.campaign_name, dd.day_name
    ORDER BY revenue DESC
'''):
    print(f"{r[0]:<24} {r[1]:<11} {r[2]:>6}  ${r[3]:>8.2f}")

print()
print("Same answer. The star schema query is simpler, faster on large data,")
print("and impossible to get the timezone wrong — it was handled once during the ETL.")
# You should see: identical results from both queries. That's the proof.


## Why Star Schema Instead of One Big Table?

> **The OLTP cash register vs OLAP report analogy in practice**

You just saw it side by side in the query above. Here's the mental model:

**One big table (OLTP query approach):**
- Every query must re-parse timestamps, re-convert timezones, re-JOIN tables
- If the timezone logic is wrong, it's wrong everywhere — and hard to find
- Adding a new "day of week" filter means touching every query that needs it
- In production at 1 billion rows, this parsing runs 1 billion times per query

**Star schema (OLAP approach):**
- Timestamps parsed and converted ONCE — in the ETL that builds fact_calls


> **Key terms:** **ETL** = Extract, Transform, Load (transform data BEFORE loading into the warehouse — clean it first, then store it). **ELT** = Extract, Load, Transform (load raw data first, then transform inside the warehouse — BigQuery/Snowflake are powerful enough to transform after loading). Modern cloud pipelines mostly use ELT because cloud warehouses handle transformation efficiently. Our medallion pipeline (Bronze → Silver → Gold) is an ELT pattern: we load raw data to Bronze (Extract + Load), then transform in Silver/Gold (Transform).

- Dimension tables pre-compute all the labels (day_name, time_period, industry)
- Analysts write simple JOINs — no functions, no CASE statements, no date math
- The query engine can skip entire partitions (folders) — reads far less data

**The trade-off:** Star schema requires upfront ETL work. But that ETL runs once per load.
The payoff: every subsequent query is simpler and faster.

| | One Big Table | Star Schema |
|---|---|---|
| **ETL effort** | None (load raw) | Higher (transform + model) |
| **Query complexity** | High (every analyst re-does the math) | Low (math done once, reused everywhere) |
| **Query performance** | Slow at scale | Fast (partition pruning + simple JOINs) |
| **Error surface** | High (timezone bugs live in every query) | Low (ETL fixes it once, queries trust it) |
| **BI tool compatibility** | Poor (tools struggle with complex SQL) | Excellent (tools love clean dimensions) |

> **Medallion reminder:** Bronze = raw. Silver = cleaned. **Gold = star schema.** Today you built the Gold layer.


## Star Schema vs. Snowflake Schema — When to Use Which?

**Star schema:** Dimensions are fully denormalized — one flat table per dimension.
**Snowflake schema:** Dimensions are normalized — sub-dimensions split into separate tables.

**Example — our call center:**

```
Star Schema:
  dim_campaign has client_name + industry in the same table

Snowflake Schema:
  dim_campaign → dim_client (separate table for client details)
```

| | Star Schema | Snowflake Schema |
|---|---|---|
| **Storage** | More (repeated values in dimension) | Less (normalized) |
| **Query simplicity** | Simpler (fewer JOINs) | More complex (extra JOINs) |
| **Query performance** | Faster (fewer JOINs, especially in columnar DBs) | Slower (more JOINs) |
| **Maintenance** | Simpler (one dimension to update) | More complex (multiple tables) |
| **Use case** | Recommended for most OLAP / data warehouses | Rare; only if dimensions are very large and values change frequently |

**Rule of thumb:** Start with star schema. Only snowflake if a dimension table is enormous (tens of millions of rows) and sub-dimensions change independently.

> In BigQuery, Redshift, and Snowflake DB, columnar storage means star schema JOINs are extremely fast. Snowflake schema's storage savings rarely justify the query complexity.

---

### Interview Questions: Schema Design

**Q1:** *"Your dim_campaign table has 50 million rows and you're adding a new 'industry vertical' field. Star or snowflake?"*
A: Depends. If the industry vertical changes independently and is referenced by multiple dimensions, snowflake makes sense. If it's just another attribute of the campaign, add it to dim_campaign (star).

**Q2:** *"A BI analyst says the dashboard is slow. You have a star schema. What do you check first?"*
A: Partitioning (are queries scanning full tables?), then clustering/indexing on the JOIN columns, then whether the dimensions are too large (consider pre-aggregating into a summary table).

**Q3:** *"What's a 'galaxy schema' (fact constellation)?"*
A: Multiple fact tables that share dimension tables. Example: fact_calls and fact_orders sharing dim_date and dim_customer. Useful when you need to analyze two related processes independently and together.


---

## 7. Partitioning & Bucketing

**One-line answer:** Partitioning splits a table into folders by a column value. The query engine skips folders it doesn't need.

### Partitioning

Imagine `fact_calls` stored on disk. Without partitioning, every query reads every row. With partitioning by date:

```
fact_calls/
  date=2026-03-10/   ← 25 rows
  date=2026-03-11/   ← 15 rows
```

A query for March 10 only reads the first folder — the engine **prunes** the March 11 partition entirely.

### Bucketing (Clustering)

Bucketing hash-distributes rows within each partition. If you frequently join on `campaign_key`, bucketing by `campaign_key` puts related rows together on disk — faster joins.

| | Partitioning | Bucketing |
|---|-------------|-----------|
| **What** | Splits table into folders by column value | Hash-distributes rows within partitions |
| **When** | Filter columns (date, region, status) | Join/group columns (customer_key, campaign_key) |
| **Benefit** | Skip entire folders (partition pruning) | Co-locate related rows for faster joins |
| **Example** | `PARTITIONED BY (date)` | `CLUSTERED BY (campaign_key)` |


> **Analogy:** Think of partitions as filing cabinets (one per month) and bucketing as organizing the folders *inside* each cabinet by customer name — so when you need “all calls from Campaign A in March,” you go straight to the right drawer within the right cabinet.

> **In SQLite, there is no physical partitioning.** In BigQuery, Spark, and Hive, it is critical for cost and performance. You will implement it in M08 (BigQuery) and M09 (Delta Lake).

In [ ]:
# --- Partitioning Simulation ---
# We're simulating partition pruning with SQLite WHERE clauses.
# In real systems (BigQuery, Spark, Hive, Delta Lake), partitioning is PHYSICAL:
# the data is stored in separate folders on disk, and the query engine skips
# entire folders that don't match the filter. That's called partition pruning.

# Full table scan — reads EVERY row regardless of date
full = cursor.execute("SELECT COUNT(*) FROM fact_calls").fetchone()[0]

# "Partitioned" query — only March 10
# WHY JOIN dim_date here? In SQLite simulation, this mimics the partition filter.
# In BigQuery, you'd write: WHERE DATE(_PARTITIONTIME) = '2026-03-10'
# and BigQuery skips all other partition folders automatically — no JOIN needed.
part = cursor.execute('''
    SELECT COUNT(*) FROM fact_calls f
    JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE dd.full_date = '2026-03-10'
''').fetchone()[0]

print("=== Partition Pruning Simulation ===")
print(f"  Full table scan:       {full} rows read")
print(f"  Partitioned (Mar 10):  {part} rows read")
print(f"  Rows skipped:          {full - part} rows ({100*(full-part)/full:.1f}% fewer reads)")

# NOTE: BigQuery charges $5 per terabyte of data scanned per query — this is real
# production pricing. Partitioning reduces the data scanned, which directly reduces your bill.
# At 1 TB, skipping 37.5% of data saves $1.88 per query. At 1,000 queries/day, that's $1,880/day.
# BigQuery cost projection
print()
print("=== BigQuery Cost Projection (1 TB table) ===")
full_cost = 5.00   # $5 per TB scanned — BigQuery on-demand pricing (as of 2026)
part_cost = round(5.00 * part / full, 2)  # proportional cost based on rows read
print(f"  Without partitioning:  ${full_cost:.2f} per query")
print(f"  With date partition:   ${part_cost:.2f} per query")
print(f"  Savings per query:     ${full_cost - part_cost:.2f}")
print(f"  Daily savings (10 queries): ${(full_cost - part_cost) * 10:.2f}")

# Clustering simulation — sorting improves locality
# WHY show GROUP BY after sorting? In a clustered table, rows with the same
# campaign_key are physically adjacent on disk. The database reads them in
# one sequential scan rather than jumping around — much faster for GROUP BY.
print()
print("=== Clustering Simulation ===")
print("Rows sorted by campaign_key for faster GROUP BY:")
print(f"{'campaign_key':>12}  {'call_count':>10}")
print("-" * 26)
for r in cursor.execute('''
    SELECT campaign_key, COUNT(*) as cnt
    FROM fact_calls
    GROUP BY campaign_key
    ORDER BY campaign_key
'''):
    print(f"{r[0]:>12}  {r[1]:>10}")

print()
print("With clustering, these groups are physically adjacent on disk.")
print("You will implement real partitioning in M08 (BigQuery) and M09 (Delta Lake).")
# You should see: 6 campaign keys (1-6), each with 6-7 calls


## Partitioning Gotchas and Interview Questions

### Common Mistakes

**1. Over-partitioning (too many small partitions)**
- Example: Partitioning by `call_id` (each call gets its own folder)
- Result: Millions of tiny files — the query engine's metadata overhead exceeds the read savings
- Rule: Each partition should hold at least a few hundred MB of data in production

**2. Wrong partition key (low cardinality column with skewed distribution)**
- Example: Partitioning by `disposition` (completed / dropped / transferred — 3 values)
- If 80% of calls are "completed," the "completed" partition is still almost the full table
- Analysts filtering by "dropped" get a tiny partition; filtering by "completed" gets nothing
- Rule: Partition by columns you actually filter on (date is almost always the right choice)

**3. Forgetting partition pruning only works with the exact partition column**
- Query: `WHERE YEAR(call_date) = 2026` on a table partitioned by `call_date`
- Many engines (including older Hive) can't prune based on a function applied to the column
- Rule: Filter on the partition column directly — `WHERE call_date BETWEEN '2026-01-01' AND '2026-12-31'`

**4. Not handling late-arriving data**
- A call from March 10 arrives in the pipeline on March 15
- Without handling, it either goes into the March 15 partition (wrong date) or fails
- Rule: Design your ETL to write to the correct partition date, not the processing date

---

### Interview Questions: Partitioning

**Q1:** *"How do you choose a partition column?"*
A: Choose the column analysts most commonly filter on (usually date/timestamp). High cardinality is fine; skewed distribution is not. The goal is partition pruning — skipping partitions the query doesn't need.

**Q2:** *"What's the difference between partitioning and clustering (bucketing)?"*
A: Partitioning splits data into physical folders by a column value — the engine skips entire folders. Clustering sorts data within a partition by a column — the engine reads fewer rows within a scanned partition. Use both together: partition by date, cluster by campaign_key.

**Q3:** *"A table is partitioned by date. A query filters by campaign_name only (no date filter). Will partition pruning help?"*
A: No. Partition pruning only applies when you filter on the partition column. Without a date filter, the engine reads all partitions. Solution: add clustering on campaign_key, or consider a secondary partition or pre-aggregated table.


---

## 8. Challenge Lab

Three challenges using the star schema you just built. Write each query, run it, and verify the results make sense.

In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 1: Top 3 Products by Revenue ---
# Hint: JOIN fact_calls + dim_product
# Hint: GROUP BY product_name and category
# Hint: ORDER BY revenue DESC, LIMIT 3
# Hint: Filter out the default row (product_key > 0) — you don't want "Unknown Product" on a revenue report

# ============ SOLUTION ============
print("=== Challenge 1: Top 3 Products by Revenue ===")
print(f"{'product_name':<30} {'category':<20} {'orders':>6} {'revenue':>10}")
print("-" * 70)
for r in cursor.execute('''
    SELECT dp.product_name, dp.category,
           SUM(f.is_converted) AS orders,
           SUM(f.order_amount) AS revenue
    FROM fact_calls f
    JOIN dim_product dp ON f.product_key = dp.product_key
    WHERE dp.product_key > 0          -- exclude default "Unknown Product" row
    GROUP BY dp.product_name, dp.category
    ORDER BY revenue DESC
    LIMIT 3                           -- top 3 only
'''):
    print(f"{r[0]:<30} {r[1]:<20} {r[2]:>6}  ${r[3]:>8.2f}")
# You should see: Summit Premium Coverage highest revenue, followed by Pinnacle Wellness Kit and Acme Widget Pro Bundle


In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 2: Conversion Rate by Time Period ---
# Which time_period has the best conversion rate?
# Hint: JOIN fact_calls to dim_time on time_key
# Hint: conv_rate = 100.0 * SUM(is_converted) / COUNT(*)
# Hint: Why 100.0 and not 100? Integer division in SQL truncates — 3/40 = 0, not 0.075
# Hint: ROUND(..., 1) gives one decimal place (e.g., 37.5%)

# ============ SOLUTION ============
print("=== Challenge 2: Best Time Period for Conversions ===")
print(f"{'time_period':<12} {'calls':>5} {'conversions':>11} {'conv_rate':>9}")
print("-" * 40)
for r in cursor.execute('''
    SELECT dt.time_period, COUNT(*) AS calls,
           SUM(f.is_converted) AS conversions,
           ROUND(100.0 * SUM(f.is_converted) / COUNT(*), 1) AS conv_rate
    FROM fact_calls f
    JOIN dim_time dt ON f.time_key = dt.time_key
    GROUP BY dt.time_period
    ORDER BY conv_rate DESC
'''):
    print(f"{r[0]:<12} {r[1]:>5} {r[2]:>11}    {r[3]:>5.1f}%")
# You should see: one time period clearly leading. Night may have 0 calls (no calls in our data after midnight local time).


In [ ]:
# TRY IT YOURSELF FIRST! Attempt the query before looking at the solution below.
# --- Challenge 3: SCD Type 2 Lookup ---
# For call_001 (caller 3135559876 on March 10), what city was the customer in?
# Hint: Get the call date from the calls table (convert UTC to EST)
# Hint: Use the SCD Type 2 pattern: WHERE caller_ani = ? AND call_date BETWEEN effective_date AND end_date
# Hint: Then verify by looking up customer_key in fact_calls and joining dim_customer

# ============ SOLUTION ============
print("=== Challenge 3: Point-in-Time Customer Lookup ===")
print()

# Get the call details — we need caller_ani and the EST date (not UTC!)
call = cursor.execute('''
    SELECT call_id, caller_ani, DATE(call_started_at, '-5 hours') AS call_date
    FROM calls WHERE call_id = 'call_001'
''').fetchone()
print(f"Call: {call[0]}, Caller: {call[1]}, Date (EST): {call[2]}")

# Look up customer using SCD Type 2 date range
# WHY not just WHERE is_current = 1? Because that gives you where they are NOW,
# not where they were WHEN THE CALL HAPPENED. Always use date range for point-in-time lookups.
cust = cursor.execute('''
    SELECT dc.customer_key, dc.city, dc.state, dc.effective_date, dc.end_date
    FROM dim_customer dc
    WHERE dc.caller_ani = ?
      AND ? BETWEEN dc.effective_date AND dc.end_date
''', (call[1], call[2])).fetchone()

print(f"Customer key: {cust[0]}, City: {cust[1]}, State: {cust[2]}")
print(f"Version active: {cust[3]} to {cust[4]}")

# Verify in fact_calls — the ETL should have used the same SCD Type 2 join
fact = cursor.execute('''
    SELECT f.call_id, f.customer_key, dc.city, dc.state
    FROM fact_calls f
    JOIN dim_customer dc ON f.customer_key = dc.customer_key
    WHERE f.call_id = 'call_001'
''').fetchone()
print()
print(f"Fact table confirms: {fact[0]} -> customer_key={fact[1]} -> {fact[2]}, {fact[3]}")
print()
print("The fact table used the SCD Type 2 date range to pick the correct customer version.")
print("On March 10, this caller was in Detroit. After March 11, they are in Chicago.")
# You should see: customer_key points to Detroit row, not Chicago row. ETL captured history correctly.


---

## 9. Key Takeaways

1. **OLTP = operations, OLAP = analytics.** Star schema is the bridge between them.
2. **Grain statement first** — everything follows from "one row = one ___." Our grain: one row = one phone call.
3. **Fact tables hold measures (numbers).** Dimension tables hold context (who, what, when, where).
4. **Surrogate keys (integers) replace natural keys (strings)** — faster joins, handles changes gracefully.
5. **Default dimension row (key = 0) eliminates NULLs** in fact table foreign keys.
6. **SCD Type 2 preserves history** — same entity, multiple rows with effective/end date ranges.
7. **Point-in-time lookup:** `WHERE date BETWEEN effective_date AND end_date` — answers "what was true THEN?"
8. **Denormalize dimensions** — flatten normalized OLTP tables (sources + clients → dim_campaign) for query simplicity.
9. **Star schema queries are simpler and faster** than equivalent OLTP queries. The ETL handles complexity once.
10. **Partitioning = physical folder structure.** The query engine skips irrelevant partitions (partition pruning).
11. **This star schema IS the Gold layer** — you will build the Bronze → Silver → Gold pipeline in M06–M16.

---

## 10. Homework & Preview

### 1. Draw the Star Schema (15 min)

On paper or in [draw.io](https://draw.io), sketch the star schema:
- `fact_calls` in the center
- 5 dimension tables around it
- Label: primary keys, foreign keys, measures, and the grain statement

### 2. SCD Type 2 Scenario (15 min)

Acme Corp changes its name to "Acme Industries" on March 15. How would you handle this in `dim_campaign`?

Write the SQL to:
1. Close the old rows (set `end_date`, `is_current = 0`)
2. Insert new rows with the updated name

*Hint: You would need to add `effective_date`, `end_date`, and `is_current` columns to `dim_campaign` first.*

### 3. Cross-Domain Design Exercise (20 min)

Design a star schema for a **hospital emergency room**:
- **Fact table:** `fact_visits` (one row = one ER visit)
- What are the **dimensions**? (who, what, when, where, how)
- What are the **measures**? (numbers you would aggregate)
- What is the **grain statement**?
- Which dimension would benefit from **SCD Type 2**? Why?

### 4. Preview M05: Hadoop & Hive

Skim these concepts — 10 minutes of reading:
- "What is Hadoop?" — distributed storage + compute
- "What is Hive?" — SQL on Hadoop

We are moving from single-machine SQL (SQLite) to **distributed data processing** across clusters of machines.

---

**Next session:** M05 — Hadoop & Hive (distributed storage, MapReduce, HiveQL)